# Reconstructing Arctic sea-ice velocity with a neural network: an observing-system-design tutorial

*AI4InSync / OAISIS — KITP 2025*

**The idea.** Some quantities are known everywhere: winds from reanalysis, ice concentration from satellites, the date, the map. Others are only measured where an instrument happens to be — sea-ice velocity from a drifting buoy, say. If we can learn how the first group relates to the second *at the instrument locations*, we can predict the second everywhere. How well that works depends on where the instruments were. That dependence is what **observing-system design** studies.

**Training.** A neural network learns the relation inputs → ice velocity *at the observation locations* — here the tracks of real Ice-Tethered Profilers, with climate-model output standing in for the measurements. Each observing system gives a different trained network: ① a single ITP (ITP 95, two months north of the Laptev Sea), ② the same ITP plus a second one elsewhere (ITP 5, six months drifting toward Fram Strait).

![](figures/training.png)

**Reconstruction.** Each trained network is then given the inputs on the *full grid* and predicts ice velocity everywhere. Because the "world" is a climate model, the true velocity is known everywhere too, so the prediction can be scored honestly (①, ②) — and the two scores compared (③): did the added observations improve the reconstruction?

![](figures/reconstruction.png)

The inputs are only ever things that would be available on the full grid in reality. The two networks in the figures are real: they are the `ITP 95` and `ITP 95 + 5` observing systems of experiment A in section 4 of this notebook.

Data: `data/oaisis_tutorial_IPSL.nc` (ships with this repository; how it was made is in `preprocessing/README.md`).

## 0. Setup

Everything the notebook needs is in this repository: one NetCDF file in `data/` and the packages in `requirements.txt`.

In [ ]:
import sys, time, warnings
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

from sklearn.linear_model import LinearRegression
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore", category=RuntimeWarning)   # nanmean over all-NaN land cells is expected
print(f"python {sys.version.split()[0]} | numpy {np.__version__} | xarray {xr.__version__} | tensorflow {tf.__version__}")

In [ ]:
DATA = Path("data/oaisis_tutorial_IPSL.nc")
ds = xr.open_dataset(DATA)
ds

## 1. Synthetic observations

### 1.1 About the data

**Ice-Tethered Profilers (ITPs)** drift with the sea ice and report their position daily, so their tracks give us a realistic *sampling pattern* of the Arctic (Toole et al., 2011; Krishfield et al., 2008; https://www.whoi.edu/itp).

We don't use the ITP measurements themselves. Instead we take daily output from CMIP6 historical simulations and pick, for every ITP position/day, the model value at the nearest grid cell and date. The result is a set of **synthetic observations** at the ITP locations where we know the truth everywhere else as well — exactly what is needed to test an observing system.

The dataset has two parts, both from the CMIP6 historical simulation of **IPSL-CM6A-LR**:

* `obs_*` — 31 ensemble members × 11 242 ITP samples (2004–2014): the synthetic observations. Dimensions `(member, sample)`, plus `obs_lat`, `obs_lon`, `obs_time` per sample.
* `map_*` — daily Arctic fields from member r1 on the model's native ocean grid, used as the truth for the reconstruction. Dimensions `(y, x, time)`.

Variables: `uas`, `vas` 10 m wind [m/s]; `sic` sea-ice concentration [%]; `siu`, `siv` sea-ice velocity [m/s]. All velocity components are **eastward / northward**. (CMIP6 stores ice velocity along the model grid's own axes, which on IPSL's tripolar grid are rotated by tens of degrees from east; the preprocessing rotated them, see `preprocessing/README.md`.) Every variable carries `long_name`, `units` and a `comment` — look at `ds` above.

In [ ]:
MEMBERS = list(ds.member.values)                 # 'r1i1p1f1' ... 'r31i1p1f1'
lat_obs = ds.obs_lat.values
lon_obs = ds.obs_lon.values
doy_obs = ds.obs_time.dt.dayofyear.values

print("members:", len(MEMBERS), "| samples:", lat_obs.size,
      "| period:", str(ds.obs_time.min().values)[:10], "to", str(ds.obs_time.max().values)[:10],
      "| latitude %.1f-%.1f N" % (lat_obs.min(), lat_obs.max()))

### 1.2 A look at the synthetic observations

Left: where the ITPs were. Right: what the model says the ice was doing there (for one ensemble member).

In [ ]:
import matplotlib.path as mpath

def polar_axes(ax, title="", lat_min=65):
    '''North-polar map with a circular boundary. Limits are set in projected coordinates
    (set_extent with a full longitude range is unreliable across cartopy versions).'''
    x, y = ax.projection.transform_point(0, lat_min, ccrs.PlateCarree())
    r = np.hypot(x, y)
    ax.set_xlim(-r, r); ax.set_ylim(-r, r)
    theta = np.linspace(0, 2 * np.pi, 200)
    ax.set_boundary(mpath.Path(np.column_stack([np.sin(theta), np.cos(theta)]) * 0.5 + 0.5), transform=ax.transAxes)
    ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=0)
    ax.coastlines(resolution="110m", linewidth=0.5)
    ax.set_title(title)

member = "r1i1p1f1"
m = ds.sel(member=member)
speed_ice = np.hypot(m.obs_siu, m.obs_siv)
speed_wind = np.hypot(m.obs_uas, m.obs_vas)

fig, axs = plt.subplots(2, 2, figsize=(11, 10), subplot_kw={"projection": ccrs.NorthPolarStereo(central_longitude=-90)}, layout="constrained")
panels = [("day of year", doy_obs, cmocean.cm.phase, {}),
          ("ice concentration [%]", m.obs_sic, cmocean.cm.ice, {"vmin": 0, "vmax": 100}),
          ("ice speed [m/s]", speed_ice, cmocean.cm.speed, {"vmin": 0, "vmax": 0.4}),
          ("wind speed [m/s]", speed_wind, cmocean.cm.speed, {"vmin": 0, "vmax": 15})]
for ax, (title, c, cmap, kw) in zip(axs.flat, panels):
    polar_axes(ax, title)
    sc = ax.scatter(lon_obs, lat_obs, c=c, s=4, cmap=cmap, transform=ccrs.PlateCarree(), **kw)
    plt.colorbar(sc, ax=ax, shrink=0.6)

In [ ]:
# relationships between inputs and targets (one member)
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (x, y, xl, yl) in zip(axs, [(m.obs_uas, m.obs_siu, "uas [m/s]", "siu [m/s]"),
                                    (m.obs_vas, m.obs_siv, "vas [m/s]", "siv [m/s]"),
                                    (speed_wind, speed_ice, "wind speed [m/s]", "ice speed [m/s]")]):
    ax.scatter(x, y, s=2, alpha=0.3)
    p = np.polyfit(x, y, 1)
    xx = np.linspace(float(x.min()), float(x.max()), 2)
    ax.plot(xx, np.polyval(p, xx), "k-", label=f"slope = {p[0]:.3f}  (r = {np.corrcoef(x, y)[0,1]:.2f})")
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.legend()
plt.tight_layout()

Ice drifts at roughly 2 % of the wind speed (the classic Nansen rule), and the correlation is high — a good sign that wind is a strong predictor. The scatter around the line is what ice concentration, season and location have to explain.

### 1.3 Building the machine-learning dataset

**Inputs (features):** `uas`, `vas`, `sic`, `doy`, `lat`, `cos(lon)`, `sin(lon)`
**Targets:** `siu`, `siv`

Longitude goes in as its cosine and sine rather than as a number: `lon/180` jumps from +1 to −1 at the dateline, and a network fed that will happily produce a jump in ice velocity across 180° W. Day of year has the same wrap-around (31 Dec → 1 Jan); with only a decade of ITPs it matters less, but try `cos`/`sin` there too.

Everything is written as a function of a dataset so that we apply *exactly the same* feature construction and scaling to the ITP samples (training) and to the full maps (reconstruction). Mixing these up is the easiest way to get a silently wrong reconstruction.

In [ ]:
FEATURES = ["uas", "vas", "sic", "doy", "lat", "cos_lon", "sin_lon"]
TARGETS = ["siu", "siv"]

def make_table(uas, vas, sic, doy, lat, lon, siu=None, siv=None):
    '''Stack broadcastable arrays into X (n, 7) and Y (n, 2). Returns Y=None if targets are missing.'''
    arrs = np.broadcast_arrays(uas, vas, sic, doy, lat, np.cos(np.radians(lon)), np.sin(np.radians(lon)))
    X = np.stack([a.ravel() for a in arrs], axis=1).astype("float32")
    Y = None
    if siu is not None:
        siu, siv = np.broadcast_arrays(siu, siv)
        Y = np.stack([siu.ravel(), siv.ravel()], axis=1).astype("float32")
    return X, Y

def obs_table(members, sample_mask=None):
    '''Feature/target table for the given ensemble members, optionally for a subset of the ITP samples.'''
    sel = slice(None) if sample_mask is None else sample_mask
    g = lambda v: ds["obs_" + v].sel(member=members).values[:, sel]      # (n_members, n_samples)
    return make_table(g("uas"), g("vas"), g("sic"), doy_obs[None, sel], lat_obs[None, sel], lon_obs[None, sel], g("siu"), g("siv"))

#### Train / validation / test split

We split by **ensemble member**, not by random sample: consecutive days along an ITP track are strongly correlated, so a random split would leak information from train to test and make the model look better than it is. Member r1 provides the reconstruction maps in section 3, so it is kept out of training.

In [ ]:
SPLIT = {"train": MEMBERS[1:21], "val": MEMBERS[21:26], "test": [MEMBERS[0]] + MEMBERS[26:31]}

X_tr, Y_tr = obs_table(SPLIT["train"])
X_va, Y_va = obs_table(SPLIT["val"])
X_te, Y_te = obs_table(SPLIT["test"])
for k, X in zip(SPLIT, (X_tr, X_va, X_te)):
    print(f"{k:5s}: {len(SPLIT[k]):2d} members ({SPLIT[k][0]} ... {SPLIT[k][-1]})  ->  {X.shape[0]:7d} samples")

#### Feature scaling

Inputs with very different ranges (wind ±20 m/s, concentration 0–100 %, latitude 70–90°) would otherwise be weighted unevenly. The winds are standardized with the **training** mean and standard deviation (using anything else would be data leakage); inputs with a natural range are simply scaled to it: `sic/100`, `doy/365`, `lat/90`, and `cos_lon`/`sin_lon` are already in [−1, 1]. Look at the scaled range of `sic` below: ITPs sit in the pack, so the network almost never sees concentrations below 0.8 — keep that in mind when we look at the ice edge later. The targets are standardized too, so the loss is in units of "standard deviations of ice velocity".

In [ ]:
class Scaler:
    '''Standardize features/targets with training statistics. Fixed scaling for doy, lat, lon.'''
    fixed = {"sic": 100.0, "doy": 365.0, "lat": 90.0, "cos_lon": 1.0, "sin_lon": 1.0}

    def fit(self, X, Y):
        self.x_mean, self.x_std = X.mean(0), X.std(0)
        self.y_mean, self.y_std = Y.mean(0), Y.std(0)
        for name, s in self.fixed.items():          # geometric inputs: divide by a constant
            i = FEATURES.index(name)
            self.x_mean[i], self.x_std[i] = 0.0, s
        return self

    def x(self, X):  return (X - self.x_mean) / self.x_std
    def y(self, Y):  return (Y - self.y_mean) / self.y_std
    def y_inv(self, Z): return Z * self.y_std + self.y_mean

scaler = Scaler().fit(X_tr, Y_tr)
x_tr, y_tr = scaler.x(X_tr), scaler.y(Y_tr)
x_va, y_va = scaler.x(X_va), scaler.y(Y_va)
x_te, y_te = scaler.x(X_te), scaler.y(Y_te)

print("scaled feature ranges (train):")
for i, f in enumerate(FEATURES):
    print(f"  {f:7s} [{x_tr[:, i].min():6.2f}, {x_tr[:, i].max():6.2f}]")

## 2. Neural network

### 2.1 Metrics

Velocity is a vector, so we score the vector error $|\mathbf{u}_{\rm pred}-\mathbf{u}_{\rm true}|$ rather than `siu` and `siv` separately — one number, and a model that gets the speed right but the direction wrong is penalised.

* **Skill** $= 1 - \mathrm{RMS}(|\mathbf{u}_{\rm pred}-\mathbf{u}_{\rm true}|)\,/\,\mathrm{RMS}(|\mathbf{u}_{\rm true}-\bar{\mathbf{u}}_{\rm true}|)$: 1 is perfect, 0 is "no better than predicting the mean velocity", negative is worse than that. It penalises bias and wrong amplitude.
* **Correlation** $r$: the vector analogue of Pearson correlation (covariance of the two vector fields over the product of their standard deviations). Pattern agreement only — a prediction can have $r \approx 1$ and still be biased or too weak. Look at both.

In [ ]:
def skill(y_true, y_pred, axis=0):
    '''Vector skill. y_* have the (u, v) components on the last axis; statistics are taken along `axis`.'''
    err = np.nansum((y_pred - y_true) ** 2, axis=-1)
    var = np.nansum((y_true - np.nanmean(y_true, axis=axis, keepdims=True)) ** 2, axis=-1)
    return 1.0 - np.sqrt(np.nanmean(err, axis=axis)) / np.sqrt(np.nanmean(var, axis=axis))

def corr(y_true, y_pred, axis=0):
    '''Vector correlation (sum of the component covariances over the product of the vector standard deviations).'''
    a = y_true - np.nanmean(y_true, axis=axis, keepdims=True)
    b = y_pred - np.nanmean(y_pred, axis=axis, keepdims=True)
    cov = np.nansum(np.nansum(a * b, axis=-1), axis=axis)
    return cov / np.sqrt(np.nansum(np.nansum(a * a, axis=-1), axis=axis) * np.nansum(np.nansum(b * b, axis=-1), axis=axis))

def report(y_true, y_pred, label=""):
    s, c = skill(y_true, y_pred), corr(y_true, y_pred)
    print(f"{label:12s} skill {s:.3f}   corr {c:.3f}")
    return s, c

# Keras versions of the same quantities, used during training
def nrmse_loss(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(y_pred - y_true))) / (tf.math.reduce_std(y_true) + 1e-7)

def corr_metric(y_true, y_pred):
    a = y_true - tf.reduce_mean(y_true); b = y_pred - tf.reduce_mean(y_pred)
    return tf.reduce_sum(a * b) / (tf.sqrt(tf.reduce_sum(a * a) * tf.reduce_sum(b * b)) + 1e-7)

### 2.2 A baseline first

Before training anything non-linear: how far does a straight line get? Ice drifts at roughly 2 % of the wind speed (the Nansen rule), so a linear regression on the same inputs is the number to beat. If the network can't improve on it, the network isn't learning anything the physics didn't already tell us.

In [ ]:
linear = LinearRegression().fit(x_tr, y_tr)
Y_lin_te = scaler.y_inv(linear.predict(x_te))
s_lin, c_lin = report(Y_te, Y_lin_te, "linear")
baseline_val_mse = float(np.mean((linear.predict(x_va) - y_va) ** 2))

### 2.3 Hyperparameters

Things you can play with: more units/layers, learning rate and epochs, dropout vs. L2 regularization, or the normalized-RMSE loss `nrmse_loss` instead of `'mse'` (same minimum, noisier curve).

In [ ]:
HP = dict(
    n_units=32,        # neurons per hidden layer
    n_layers=2,        # hidden layers
    dropout=0.0,       # dropout rate (0 = off)
    l2=1e-3,           # L2 weight penalty
    lr=1e-3,           # learning rate
    epochs=30,
    batch_size=256,
    activation="relu",
    loss="mse",        # or nrmse_loss
)

In [ ]:
def build_model(n_in, n_out, n_units, n_layers, dropout, l2, lr, activation, loss, **_):
    model = keras.Sequential([layers.Input(shape=(n_in,))])
    for _ in range(n_layers):
        model.add(layers.Dense(n_units, activation=activation, kernel_regularizer=keras.regularizers.l2(l2)))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(n_out))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=loss, metrics=[corr_metric])
    return model

def train_model(x_tr, y_tr, x_va, y_va, hp=HP, verbose=0, seed=0):
    keras.utils.set_random_seed(seed)
    model = build_model(x_tr.shape[1], y_tr.shape[1], **hp)
    hist = model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=hp["epochs"],
                     batch_size=hp["batch_size"], shuffle=True, verbose=verbose)
    return model, hist.history

def plot_history(h, baseline=None):
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, k, lab in zip(axs, ["loss", "corr_metric"], ["loss", "correlation"]):
        ax.plot(h[k], label="train"); ax.plot(h["val_" + k], label="validation")
        ax.set_xlabel("epoch"); ax.set_ylabel(lab)
    if baseline is not None:
        axs[0].axhline(baseline, color="k", ls="--", lw=1, label="linear regression (validation)")
    axs[0].set_ylim(0, None); axs[0].legend(); axs[1].legend()
    plt.tight_layout()

### 2.4 Train

The loss axis starts at zero on purpose: most of the way from "predict the mean" (loss 1) to the final loss is covered by the straight line, and the network's contribution is the small gap below the dashed line. That is what this problem looks like — same-day wind, concentration and position don't determine ice drift beyond that (ocean currents, internal ice stress and inertia are not inputs).

In [ ]:
model, history = train_model(x_tr, y_tr, x_va, y_va, verbose=1)
model.summary()
plot_history(history, baseline=baseline_val_mse)

### 2.5 Evaluate on the held-out test members

The model has never seen these ensemble members. Note that we evaluate in *physical* units (m/s) after undoing the scaling.

In [ ]:
Y_pred_te = scaler.y_inv(model.predict(x_te, verbose=0))
s_te, c_te = report(Y_te, Y_pred_te, "test")

fig, axs = plt.subplots(1, 3, figsize=(15, 4.5), layout="constrained")
for ax, k, name in zip(axs, range(2), TARGETS):
    ax.hexbin(Y_te[:, k], Y_pred_te[:, k], gridsize=60, bins="log", cmap="Greys")
    ax.plot([-0.6, 0.6], [-0.6, 0.6], "r-", lw=1)
    ax.set_xlabel(f"true {name} [m/s]"); ax.set_ylabel(f"predicted {name} [m/s]"); ax.set_aspect("equal")
# where along the ITP tracks is the vector error largest? (one test member)
err = np.hypot(*(Y_pred_te - Y_te).reshape(len(SPLIT["test"]), -1, 2)[0].T)
axs[2].remove(); ax = fig.add_subplot(1, 3, 3, projection=ccrs.NorthPolarStereo(central_longitude=-90))
polar_axes(ax, "|predicted - true| [m/s]")
sc = ax.scatter(lon_obs, lat_obs, c=err, s=4, cmap=cmocean.cm.amp, vmin=0, vmax=0.1, transform=ccrs.PlateCarree())
plt.colorbar(sc, ax=ax, shrink=0.7)
fig.suptitle(f"test members: skill {s_te:.2f}, r = {c_te:.2f}")

## 3. Reconstruction: from ITP tracks to the whole Arctic

The inputs (wind, concentration, date, position) are known at every grid cell of the model. We feed them through the network trained on the ITP samples and compare the result with the model's own ice velocity — which the network has never seen outside the ITP tracks.

The `map_*` variables hold IPSL-CM6A-LR member r1 on its native ocean grid (`y` × `x` × `time`), north of 55 °N, NaN over land. Wind has already been put on the ice grid.

In [ ]:
lat_map, lon_map = ds.map_lat.values, ds.map_lon.values
doy_map = ds.map_time.dt.dayofyear.values
sic_map, uas_map, vas_map, siu_map, siv_map = (ds["map_" + v].values for v in ("sic", "uas", "vas", "siu", "siv"))
ny, nx, nt = siu_map.shape
print(f"grid {ny} x {nx}, {nt} days: {str(ds.map_time[0].values)[:10]} to {str(ds.map_time[-1].values)[:10]}")

# ocean cells with ice: this is where a reconstruction is meaningful
ice_mask = np.isfinite(siu_map).all(axis=2) & (np.nanmean(sic_map, axis=2) > 15)
print(f"ice-covered ocean cells: {ice_mask.sum()} of {ny*nx}")

### 3.1 Plotting on a curvilinear grid

IPSL-CM6A-LR uses the NEMO tripolar ORCA1 grid. If you hand `pcolormesh` longitudes with `transform=PlateCarree()`, the cells that straddle the dateline (one per grid row) are drawn as long streaks across the map. Projecting the cell coordinates to the map's x/y *first* and drawing in those coordinates avoids the problem: in a polar stereographic plane there is no dateline. The last grid row is the "north fold" of the tripolar grid, where the grid meets its own mirror image; appending that mirror closes the gap.

In [ ]:
def polar_pcolormesh(ax, lon2d, lat2d, field, fold=True, **kw):
    '''pcolormesh on a curvilinear lat/lon grid, drawn in the map projection's own coordinates (no dateline streaks).
    fold=True appends the mirror image of the last row (ORCA north fold) so there is no gap along the fold line.'''
    if fold:
        perm = (lon2d.shape[1] - 2 - np.arange(lon2d.shape[1])) % lon2d.shape[1]   # cell j <-> cell (n-2-j) across the fold
        lon2d, lat2d, field = (np.vstack([a, a[-1:, perm]]) for a in (lon2d, lat2d, field))
    xyz = ax.projection.transform_points(ccrs.PlateCarree(), lon2d, lat2d)
    return ax.pcolormesh(xyz[..., 0], xyz[..., 1], np.ma.masked_invalid(field), shading="auto", **kw)

t = 0
fig, axs = plt.subplots(2, 3, figsize=(15, 9), subplot_kw={"projection": ccrs.NorthPolarStereo(central_longitude=-90)}, layout="constrained")
fields = [("siu [m/s]", siu_map[:, :, t], cmocean.cm.balance, dict(vmin=-0.3, vmax=0.3)),
          ("siv [m/s]", siv_map[:, :, t], cmocean.cm.balance, dict(vmin=-0.3, vmax=0.3)),
          ("sic [%]", sic_map[:, :, t], cmocean.cm.ice, dict(vmin=0, vmax=100)),
          ("uas [m/s]", uas_map[:, :, t], cmocean.cm.balance, dict(vmin=-12, vmax=12)),
          ("vas [m/s]", vas_map[:, :, t], cmocean.cm.balance, dict(vmin=-12, vmax=12)),
          ("ITP sampling", None, None, {})]
for ax, (title, f, cmap, kw) in zip(axs.flat, fields):
    polar_axes(ax, title)
    if f is None:
        ax.scatter(lon_obs, lat_obs, s=1, c="k", transform=ccrs.PlateCarree())
    else:
        pm = polar_pcolormesh(ax, lon_map, lat_map, f, cmap=cmap, **kw)
        plt.colorbar(pm, ax=ax, shrink=0.7)
plt.suptitle(f"IPSL-CM6A-LR r1, {str(ds.map_time[t].values)[:10]}")

### 3.2 Reconstruct

Same `make_table` → same `scaler` → same model. The only new thing is reshaping the flat prediction back to `(ny, nx, nt, 2)`.

In [ ]:
X_map, Y_map = make_table(uas_map, vas_map, sic_map, doy_map[None, None, :], lat_map[:, :, None], lon_map[:, :, None], siu_map, siv_map)
ok = np.isfinite(X_map).all(axis=1)              # land cells have NaN inputs

def reconstruct(model, X_map):
    pred = np.full((X_map.shape[0], 2), np.nan, dtype="float32")
    pred[ok] = scaler.y_inv(model.predict(scaler.x(X_map[ok]), verbose=0))
    return pred.reshape(ny, nx, nt, 2)

Y_true_map = Y_map.reshape(ny, nx, nt, 2)
Y_rec = reconstruct(model, X_map)

### 3.3 Skill of the reconstruction

Two views:

* **Pooled** over all ice-covered cells and days: one number, directly comparable to the test score in section 2.
* **Per grid cell** over the `nt` days: a map of where the reconstruction works. With only a few days per cell the per-cell numbers are noisy and a small bias is enough to push skill below zero, so read the map qualitatively; cells with no ice or no variability are masked.

In [ ]:
def pooled_scores(Y_true, Y_rec, mask=ice_mask, label="pooled"):
    yt, yp = Y_true[mask].reshape(-1, 2), Y_rec[mask].reshape(-1, 2)
    return report(yt, yp, label)

def cell_scores(Y_true, Y_rec, mask=ice_mask, min_std=0.01):
    '''Per-cell vector skill and correlation (ny, nx) over time, masked where the truth barely varies.'''
    s, c = skill(Y_true, Y_rec, axis=2), corr(Y_true, Y_rec, axis=2)
    low = (np.sqrt(np.nanmean(np.nansum((Y_true - np.nanmean(Y_true, axis=2, keepdims=True)) ** 2, axis=-1), axis=2)) < min_std) | ~mask
    s[low] = np.nan; c[low] = np.nan
    return s, c

def plot_cell_scores(s, c, title=""):
    fig, axs = plt.subplots(1, 2, figsize=(10, 5), subplot_kw={"projection": ccrs.NorthPolarStereo(central_longitude=-90)}, layout="constrained")
    for ax, f, lab in zip(axs, (s, c), ("skill", "correlation")):
        polar_axes(ax, lab)
        pm = polar_pcolormesh(ax, lon_map, lat_map, f, cmap=cmocean.cm.balance, vmin=-1, vmax=1)
        plt.colorbar(pm, ax=ax, shrink=0.7)
    fig.suptitle(title)

pooled_scores(Y_true_map, Y_rec)
s_full, c_full = cell_scores(Y_true_map, Y_rec)
plot_cell_scores(s_full, c_full, "Reconstruction trained on all ITP samples")

The reconstruction scores well below the test score of section 2 — and that is the point of the exercise, not a bug. The ITPs live in thick, slow pack ice (concentration ≈ 95 %, drift σ ≈ 0.09 m/s); the maps also contain the marginal ice zone and early-winter coastal ice (σ ≈ 0.16 m/s; Fram Strait, Barents and Bering Seas, Baffin Bay) that no ITP has ever sampled. The network has to extrapolate there. Where the error below is large is where new observations would be most valuable.

In [ ]:
# the reconstruction itself vs. truth, one day: speed, and the magnitude of the vector error
t = 0
speed_true = np.hypot(Y_true_map[:, :, t, 0], Y_true_map[:, :, t, 1])
speed_rec = np.hypot(Y_rec[:, :, t, 0], Y_rec[:, :, t, 1])
err_mag = np.hypot(*(Y_rec[:, :, t] - Y_true_map[:, :, t]).transpose(2, 0, 1))
fig, axs = plt.subplots(1, 3, figsize=(15, 5), subplot_kw={"projection": ccrs.NorthPolarStereo(central_longitude=-90)}, layout="constrained")
for ax, (f, lab, cmap, kw) in zip(axs, [(speed_true, "true ice speed [m/s]", cmocean.cm.speed, dict(vmin=0, vmax=0.3)),
                                        (speed_rec, "reconstructed ice speed [m/s]", cmocean.cm.speed, dict(vmin=0, vmax=0.3)),
                                        (err_mag, "|reconstruction - truth| [m/s]", cmocean.cm.amp, dict(vmin=0, vmax=0.15))]):
    polar_axes(ax, lab)
    pm = polar_pcolormesh(ax, lon_map, lat_map, np.where(ice_mask, f, np.nan), cmap=cmap, **kw)
    plt.colorbar(pm, ax=ax, shrink=0.7)
fig.suptitle(str(ds.map_time[t].values)[:10])

## 4. Observing-system design: what if we had different observations?

Everything above is wrapped into two functions. `run_experiment` takes a dictionary of *observing systems* — each a selection of ITP samples — retrains the network on each (several seeds, because one training run is noisy), scores the Arctic-wide reconstruction, and draws the results. `run_sequence` does the same for observing systems that grow one ITP at a time. Two experiments here, and a template for your own in section 5:

* **A — where.** One ITP at a time, in different parts of the Arctic. How much does the location of a single instrument matter?
* **B — how many.** Start from one ITP and add one at a time. How does skill grow, and how many ITPs does it take to approach the skill of using all of them?

### 4.1 Selecting observations

An observing system is a boolean array over the 11 242 ITP samples: `True` for the samples we get to use. A few selectors build these arrays from questions you would actually ask; combine them with `&` (and), `|` (or) and `~` (not).

| selector | keeps |
|---|---|
| `tracks(5, 15)` | the whole tracks of these ITPs (ids on the map below) |
| `region(lon=(-180, -120), lat=(70, 80))` | samples inside a lon/lat box (dateline-safe) |
| `near(lat=77, lon=-138, radius_km=75)` | samples within a circle — a stationary "mooring" |
| `years(2007, 2010)`, `season("JJA")`, `months(3, 4)` | samples in those years / months |
| `every(5)` | every 5th sample (thins the daily record) |
| `everything()` | all samples — the reference every experiment is compared with |

In [ ]:
track_obs = ds.obs_track.values
track_ids, track_len = np.unique(track_obs, return_counts=True)
n_samples = lat_obs.size
year_obs, month_obs = ds.obs_time.dt.year.values, ds.obs_time.dt.month.values

def everything():
    return np.ones(n_samples, bool)

def tracks(*ids):
    return np.isin(track_obs, ids)

def region(lon=(-180, 180), lat=(-90, 90)):
    in_lon = ((lon_obs - lon[0]) % 360) <= ((lon[1] - lon[0]) % 360 or 360)
    return in_lon & (lat_obs >= lat[0]) & (lat_obs <= lat[1])

def near(lat, lon, radius_km=75):
    la0, lo0, la, lo = map(np.radians, (lat, lon, lat_obs, lon_obs))
    d = 2 * 6371 * np.arcsin(np.sqrt(np.sin((la - la0) / 2) ** 2 + np.cos(la0) * np.cos(la) * np.sin((lo - lo0) / 2) ** 2))
    return d <= radius_km

def years(first, last=None):
    return (year_obs >= first) & (year_obs <= (last or first))

def months(*m):
    return np.isin(month_obs, m)

def season(name):
    return months(*{"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11)}[name])

def every(n):
    return np.arange(n_samples) % n == 0

def describe(sel):
    '''One line: how many samples, tracks and years a selection covers.'''
    t = ds.obs_time.values[sel]
    return f"{sel.sum():6d} samples, {len(np.unique(track_obs[sel])):3d} tracks, {str(t.min())[:4]}-{str(t.max())[:4]}" if sel.any() else "  empty selection"

The ITP tracks, labelled by id (only those with at least 30 days of data), and the reference observing system — everything.

In [ ]:
long_tracks = track_ids[track_len >= 30]
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={"projection": ccrs.NorthPolarStereo(central_longitude=-90)})
polar_axes(ax, f"{len(long_tracks)} ITP tracks with >= 30 days (of {len(track_ids)})", lat_min=72)
cmap = plt.get_cmap("tab20")
for i in long_tracks:
    m = track_obs == i
    ax.scatter(lon_obs[m], lat_obs[m], s=2, color=cmap(i % 20), transform=ccrs.PlateCarree())
    ax.text(lon_obs[m][0], lat_obs[m][0], str(i), fontsize=7, transform=ccrs.PlateCarree(), ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
print("everything():", describe(everything()))

### 4.2 Running an experiment

`run_experiment(name, scenarios)` trains one network per scenario and per seed, always adds the `all` reference, and draws three figures: the sampling maps, the skill bar chart (bar = mean over seeds, whisker = min–max; differences smaller than the whiskers are not results), and maps of where each scenario falls short of the reference. `quick=True` uses one seed and fewer epochs — good for a first look before a full run. Every result is cached: re-running an unchanged selection returns it instantly and says so, which is why the `all` reference is trained only once.

In [ ]:
SEEDS = (0, 1, 2)
HP_OSD = dict(HP, epochs=15)       # the training curve is flat by then; keeps the loops short
_cache = {}

def train_on(sel, name, seeds=SEEDS, hp=HP_OSD):
    '''Retrain on a selection of ITP samples (once per seed); pooled skills and the seed-mean skill map.
    Results are cached, so re-running the same selection with the same settings returns instantly.'''
    key = (sel.tobytes(), seeds, hp["epochs"])
    if key in _cache:
        res = dict(_cache[key], name=name)
        print(f"{name:14s} {describe(sel)}   skill {res['skills'].mean():.3f}  (cached)")
        return res
    t0 = time.time()
    tabs = {split: obs_table(mem, sel) for split, mem in SPLIT.items()}
    sc = Scaler().fit(*tabs["train"])
    skills, maps = [], []
    for seed in seeds:
        mdl, _ = train_model(sc.x(tabs["train"][0]), sc.y(tabs["train"][1]), sc.x(tabs["val"][0]), sc.y(tabs["val"][1]), hp=hp, seed=seed)
        pred = np.full((X_map.shape[0], 2), np.nan, dtype="float32")
        pred[ok] = sc.y_inv(mdl.predict(sc.x(X_map[ok]), verbose=0))
        Y_rec_s = pred.reshape(ny, nx, nt, 2)
        skills.append(skill(Y_true_map[ice_mask].reshape(-1, 2), Y_rec_s[ice_mask].reshape(-1, 2)))
        maps.append(cell_scores(Y_true_map, Y_rec_s)[0])
    res = dict(name=name, sel=sel, n_obs=int(sel.sum()), skills=np.array(skills), skill_map=np.nanmean(maps, axis=0))
    _cache[key] = res
    print(f"{name:14s} {describe(sel)}   skill {res['skills'].mean():.3f}  (seeds: {' '.join(f'{v:.3f}' for v in skills)})   "
          f"{len(seeds)} network{'s' if len(seeds) > 1 else ''} x {hp['epochs']} epochs in {time.time() - t0:.0f} s")
    return res

def run_experiment(name, scenarios, quick=False, maps=True):
    '''Train and score every observing system in `scenarios` (plus the `all` reference); draw sampling maps, skill bars, difference maps.'''
    seeds, hp = ((0,), dict(HP_OSD, epochs=10)) if quick else (SEEDS, HP_OSD)
    scenarios = {**scenarios, "all": everything()}
    print(f"{name}: {len(scenarios)} observing systems x {len(seeds)} seed(s)\n")
    results = {k: train_on(v, k, seeds, hp) for k, v in scenarios.items()}
    plot_sampling(scenarios, title=name)
    plot_skill_bars(results, title=name)
    if maps:
        plot_skill_difference(results)
    return results

def run_sequence(name, sequence, quick=False, labels=None, map_steps=None):
    '''Observing systems that grow one ITP at a time: skill curve against the `all` reference, plus maps at `map_steps`.'''
    seeds, hp = ((0,), dict(HP_OSD, epochs=10)) if quick else (SEEDS, HP_OSD)
    print(f"{name}: {len(sequence)} steps x {len(seeds)} seed(s); order of addition {list(sequence)}\n")
    steps = [train_on(tracks(*sequence[:k]), f"{k} ITP{'s' if k > 1 else ''}", seeds, hp) for k in range(1, len(sequence) + 1)]
    ref = train_on(everything(), "all", seeds, hp)
    plot_skill_curve(steps, ref, title=name, labels=labels or {})
    if map_steps:
        plot_skill_difference({"all": ref, **{steps[k - 1]["name"]: steps[k - 1] for k in map_steps}})
    return steps

In [ ]:
# --- plotting for experiments -------------------------------------------------------------------------
def plot_sampling(scenarios, title="", lat_min=70):
    fig, axs = plt.subplots(1, len(scenarios), figsize=(3.6 * len(scenarios), 3.8), subplot_kw={"projection": ccrs.NorthPolarStereo(central_longitude=-90)}, layout="constrained")
    for ax, (label, sel) in zip(np.atleast_1d(axs), scenarios.items()):
        polar_axes(ax, label, lat_min=lat_min)
        ax.scatter(lon_obs[~sel], lat_obs[~sel], s=1, c="lightgray", transform=ccrs.PlateCarree())
        ax.scatter(lon_obs[sel], lat_obs[sel], s=1, c="k", transform=ccrs.PlateCarree())
    fig.suptitle(title)

def plot_skill_bars(results, title="", reference="all"):
    names = list(results)
    mean = np.array([results[n]["skills"].mean() for n in names])
    lo = mean - np.array([results[n]["skills"].min() for n in names]); hi = np.array([results[n]["skills"].max() for n in names]) - mean
    fig, ax = plt.subplots(figsize=(1.6 * len(names) + 2, 4))
    ax.bar(names, mean, yerr=[lo, hi], capsize=4, color=["#333333" if n == reference else "#b4a7d6" for n in names], width=0.6)
    for i, n in enumerate(names):
        ax.text(i, 0.02, f"{results[n]['n_obs']:,}", ha="center", va="bottom", color="white", fontsize=9)
    ax.set_ylabel("reconstruction skill"); ax.set_ylim(0, max(0.8, mean.max() + 0.1))
    ax.set_title(f"{title}   (number of samples on each bar)", fontsize=10)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)

def plot_skill_difference(results, reference="all", vmax=0.5):
    base = results[reference]["skill_map"]
    others = [r for r in results.values() if r["name"] != reference]
    fig, axs = plt.subplots(1, len(others), figsize=(4.4 * len(others), 5), subplot_kw={"projection": ccrs.NorthPolarStereo(central_longitude=-90)}, layout="constrained")
    for ax, r in zip(np.atleast_1d(axs), others):
        polar_axes(ax, f"{reference} - {r['name']}")
        pm = polar_pcolormesh(ax, lon_map, lat_map, base - r["skill_map"], cmap=cmocean.cm.balance, vmin=-vmax, vmax=vmax)
        plt.colorbar(pm, ax=ax, shrink=0.7)
    fig.suptitle("skill of the reference minus skill of the scenario: red = the missing observations were helping there", fontsize=10)

def plot_skill_curve(steps, ref, title="", labels={}):
    n = np.arange(1, len(steps) + 1)
    mean = np.array([s["skills"].mean() for s in steps])
    lo = np.array([s["skills"].min() for s in steps]); hi = np.array([s["skills"].max() for s in steps])
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.fill_between(n, lo, hi, color="#b4a7d6", alpha=0.4, lw=0, label="min-max over seeds")
    ax.plot(n, mean, "o-", color="#5b4b8a", label="skill (seed mean)")
    ax.axhline(ref["skills"].mean(), color="#333333", ls="--", lw=1.2, label=f"all ITP samples ({ref['n_obs']:,})")
    for k, lab in labels.items():
        ax.annotate(lab, (k, mean[k - 1]), textcoords="offset points", xytext=(0, -18), ha="center", fontsize=10)
    ax.set_xticks(n); ax.set_xlabel("number of ITPs used for training"); ax.set_ylabel("reconstruction skill")
    ax.set_ylim(min(0.35, lo.min() - 0.05), 0.65); ax.legend(loc="lower right", frameon=False); ax.set_title(title, fontsize=10)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)

### 4.3 Experiment A — where: one ITP at a time

Three ITPs that drifted through different parts of the Arctic, each used alone — plus the pair from the schematic at the top of the notebook (NN 1 = ITP 95, NN 2 = ITP 95 + ITP 5).

In [ ]:
results_A = run_experiment("A: where", {
    "ITP 95": tracks(95),      # two months north of the Laptev Sea
    "ITP 5":  tracks(5),       # six months drifting toward Fram Strait
    "ITP 15": tracks(15),      # six months north of the Canadian Archipelago
    "ITP 95 + 5": tracks(95, 5),   # NN 1 -> NN 2 of the schematic: the first ITP plus a second one elsewhere
})

Where the single ITP was matters as much as how long it lasted: ITP 95 has a third of the samples of ITP 5 and reaches about the same skill. Adding ITP 5 to ITP 95 (NN 1 → NN 2 of the schematic) lifts the skill well above either ITP alone: two regions beat one. The difference maps show *where* each network falls short of the full reconstruction.

### 4.4 Experiment B — how many: adding ITPs one at a time

Start from one ITP in the Beaufort Sea and add one track at a time: first the other long Beaufort tracks, then — following a simple design rule — always the long track that is *farthest from everything we already observe*. The dashed line is the skill with all ITP samples.

In [ ]:
def centroid(ids):
    m = tracks(*ids); la, lo = np.radians(lat_obs[m]), np.radians(lon_obs[m])
    return np.array([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)]).mean(1)

def mean_lon(i):
    lo = np.radians(lon_obs[track_obs == i]); return np.degrees(np.arctan2(np.sin(lo).mean(), np.cos(lo).mean()))

big = track_ids[track_len >= 100]
beaufort = [int(i) for i in big if mean_lon(i) < -120][:4]
candidates = sorted([int(i) for i in big if mean_lon(i) > -90], key=lambda i: -track_len[track_ids == i][0])[:4]
elsewhere = []
while candidates:                                   # greedy: next, the candidate farthest from what we have
    have = centroid(beaufort + elsewhere)
    far = max(candidates, key=lambda i: np.linalg.norm(centroid([i]) - have))
    elsewhere.append(far); candidates.remove(far)
sequence = beaufort + elsewhere + [5, 15]          # 10 steps

results_B = run_sequence("B: how many", sequence, map_steps=(1, 4, 8, 10))

Two things to read off this curve (the axis is zoomed; the changes are larger than the seed spread). First, the three extra Beaufort tracks (steps 2–4, +1 500 samples) buy almost nothing, while the first tracks from *other* regions (steps 5–7, +1 000 samples) lift skill to the all-ITP line: a new region is worth more than more data from a region we already observe. Second, six or seven well-placed ITPs — about a quarter of the samples — do as well as all 105 tracks, and adding more (steps 9–10) changes nothing. That is the answer to "what is the minimum observing system?" for this problem, with the caveat that the dashed line is not an upper bound: the full set is itself heavily concentrated in the Beaufort Gyre. The maps show the reconstruction filling in as ITPs are added.

## 5. Design your own experiment

An experiment is a dictionary: each entry is a **label** (any text — it becomes the bar's name) and a **selection** of ITP samples. To make your own, edit the dictionary in the cell below and run it.

**1. Choose what to select from.** Every selector returns `True`/`False` for each of the 11 242 ITP samples. What you can pick:

| you want … | write | what you can put in |
|---|---|---|
| particular ITPs | `tracks(5, 15)` | any track ids from the map in section 4.1 (1–105; the 45 labelled ones have ≥ 30 days) |
| a region | `region(lon=(-180, -120), lat=(75, 85))` | longitudes −180…180 (a box may cross the dateline, e.g. `lon=(150, -150)`), latitudes 71–90; leave out `lon=` or `lat=` for no limit |
| one spot | `near(lat=77.1, lon=-138, radius_km=75)` | any position; the radius sets how "stationary" the mooring is |
| a period | `years(2007)`, `years(2007, 2010)` | 2004–2014 |
| a time of year | `season("JJA")`, `months(3, 4)` | `"DJF"`, `"MAM"`, `"JJA"`, `"SON"`, or month numbers 1–12 |
| thinner sampling | `every(5)` | any integer: keep every n-th sample |
| everything | `everything()` | the reference — added to every experiment automatically |

**2. Combine them.** `&` means *both*, `|` means *either*, `~` means *not*: `region(lon=(-180, -120)) & season("JJA")` is the Beaufort Sea in summer; `~tracks(1, 2, 3, 4)` is everything except those four ITPs; `tracks(22) & years(2010)` is one year of one ITP. Check a selection with `describe(sel)` before training on it — a selection with fewer than ~50 samples will train but tell you little.

**3. Run it.** `run_experiment("a name", my_experiment)` trains one network per entry (plus the reference), prints a table and draws three figures: where the samples are, the skill of each system, and maps of where each falls short of the reference. Use `quick=True` first (one seed, ~1 minute); drop it for the full three-seed run (~1 minute per entry) before drawing conclusions — the whiskers tell you which differences are real. For a system that grows step by step, give `run_sequence("a name", [ids in order of addition])` a list of track ids instead.

The example below asks whether a stationary mooring or a drifting buoy, with a similar number of days, gives the better reconstruction.

In [ ]:
my_experiment = {
    "mooring, 9 yr":     near(lat=77.1, lon=-138, radius_km=75),      # all ITP days within 75 km of one spot (11 ITPs, 594 days)
    "one buoy, 2 yr":    tracks(22),                                   # the longest single ITP track (746 days)
    "Beaufort, summer":  region(lon=(-180, -120)) & season("JJA"),
    "north of 85 N":     region(lat=(85, 90)),
}
for label, sel in my_experiment.items():
    print(f"{label:18s} {describe(sel)}")

my_results = run_experiment("my experiment", my_experiment, quick=True)

Some questions to try, each one line in the dictionary:

* **Thinning.** `every(5)` against `everything()`: is a daily record needed, or would weekly do? (`every(2)`, `every(10)` …)
* **Seasons.** `season("JJA")` vs `season("DJF")`: the maps are from November–December — does it help to have observed in the same season?
* **Years.** `years(2004, 2008)` vs `years(2010, 2014)`: do early and late ITPs reconstruct the same 2000 field equally well?
* **Regions.** `region(lon=(-180, -120))` (Beaufort only), `~region(lon=(-180, -120))` (everything but the Beaufort), `region(lat=(85, 90))` (the central Arctic only).
* **Mooring vs. buoy.** Change `radius_km`, or pick another spot (`near(lat=83.8, lon=-111)` has 3 tracks over 4.6 years). Then match the number of days: `near(...)` vs `tracks(22) & years(2007, 2008)`.
* **Order matters.** Re-run experiment B with `sequence = beaufort + [7, 27, 24, 96, 5, 15]` (the same tracks, longest first instead of farthest first). Skill *drops* at step 5, with a large spread between seeds, before recovering at step 6. Why would adding one track near the pole to four Beaufort tracks make things worse? (Hint: look at where track 7 is, and at what the network has to do between the two clusters.)
* **Is the network needed?** Replace `train_model` with `LinearRegression` inside `train_on`. Do the rankings change?
* **How many seeds are enough?** `SEEDS = (0, 1, 2, 3, 4)` for two scenarios that looked different. Are they still?
* **Inputs.** Drop `sic`, or drop `lat`/`lon`, from `FEATURES`. What does each contribute, and what does that say about what the ITPs are actually "observing"?